Importing everything.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.pipeline import Pipeline


data = pd.read_csv("IT_customer_churn.csv")

In [2]:
data.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [3]:
data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
for col in data.columns:
    print(f"--- {col} ---")
    print(data[col].value_counts(dropna=False)) 


--- gender ---
gender
Male      3555
Female    3488
Name: count, dtype: int64
--- SeniorCitizen ---
SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64
--- Partner ---
Partner
No     3641
Yes    3402
Name: count, dtype: int64
--- Dependents ---
Dependents
No     4933
Yes    2110
Name: count, dtype: int64
--- tenure ---
tenure
1     613
72    362
2     238
3     200
4     176
     ... 
28     57
39     56
44     51
36     50
0      11
Name: count, Length: 73, dtype: int64
--- PhoneService ---
PhoneService
Yes    6361
No      682
Name: count, dtype: int64
--- MultipleLines ---
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64
--- InternetService ---
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64
--- OnlineSecurity ---
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64
--- OnlineBackup ---


In [5]:
X_train, X_test, y_train, y_test = train_test_split(data.drop("Churn", axis=1), data["Churn"], test_size=0.2, random_state=42)

The most important cell in the whole notebook as it includes the preprocessing and pipeline.

In [6]:
ordinal_col = ["PaperlessBilling", "StreamingMovies", "StreamingTV", 
               "TechSupport", "DeviceProtection", "OnlineBackup", "OnlineSecurity", 
               "PhoneService", "Partner", "gender", "MultipleLines", "Dependents"]
encoder_col = ["PaymentMethod", "Contract", "InternetService"]
preprocessor = ColumnTransformer(
    transformers=[
        ("Ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ordinal_col),
        ("HotEncoder", OneHotEncoder(handle_unknown="ignore"), encoder_col)
    ],
    remainder="passthrough"
)


X_train["TotalCharges"] = pd.to_numeric(X_train["TotalCharges"], errors="coerce")
X_test["TotalCharges"] = pd.to_numeric(X_test["TotalCharges"], errors="coerce")


median_val = X_train["TotalCharges"].median()
X_train["TotalCharges"] = X_train["TotalCharges"].fillna(median_val)
X_test["TotalCharges"] = X_test["TotalCharges"].fillna(median_val)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        max_features=0.5,
        max_depth=10,
        min_samples_split=4,
        class_weight="balanced"
    ))
])

le = LabelEncoder()
y_train2 = le.fit_transform(y_train)
y_test2 = le.transform(y_test)


In [7]:
#model = RandomForestClassifier(n_estimators=100, random_state=42, max_features=0.5, max_depth=10,
                                #min_samples_split=4, class_weight="balanced")
model.fit(X_train, y_train2)
prediction = model.predict(X_test)

THRESHOLD = 0.40
y_probs = model.predict_proba(X_test)[:, 1]
prediction = (y_probs >= THRESHOLD).astype(int)

AS = accuracy_score(y_true=y_test2,y_pred=prediction)
MAE = mean_absolute_error(y_true=y_test2, y_pred=prediction)

CR = classification_report(y_test2, prediction)

print(f"Accuracy score: {AS}\nMean Absolute Error: {MAE}\nClassification Report: {CR}")




Accuracy score: 0.7374024130589071
Mean Absolute Error: 0.262597586941093
Classification Report:               precision    recall  f1-score   support

           0       0.93      0.69      0.79      1036
           1       0.50      0.86      0.64       373

    accuracy                           0.74      1409
   macro avg       0.72      0.78      0.72      1409
weighted avg       0.82      0.74      0.75      1409



The recall is high though the accuracy is low. (This state is preferable for industries.)

In [9]:
import joblib

joblib.dump(model, "churn_model")

['churn_model']